# P4-4 · Bài 7 · Agent

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q langchain-anthropic langchain-core langgraph

## Lý thuyết

In [ ]:
def run_agent(user_msg, tools, tool_fns, max_steps=8):
    messages = [{"role": "user", "content": user_msg}]
    for step in range(max_steps):
        resp = client.messages.create(model="claude-haiku-5-5", max_tokens=1024,
                                      system=SYSTEM, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        calls = [b for b in resp.content if b.type == "tool_use"]
        if not calls:
            return resp.content[0].text, messages                  # xong việc
        results = []
        for c in calls:
            try:
                out = tool_fns[c.name](**c.input)
            except Exception as e:                                 # lỗi tool trả lại cho model, không làm sập vòng lặp
                out = f"Lỗi khi gọi {c.name}: {e}"
            results.append({"type": "tool_result", "tool_use_id": c.id, "content": str(out)[:4000]})
        messages.append({"role": "user", "content": results})
    return "Chưa hoàn thành sau số bước tối đa, đã chuyển cho nhân viên.", messages

In [ ]:
# Cần ANTHROPIC_API_KEY: chạy trên máy bạn hoặc Colab
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool

@tool
def get_order_status(order_id: str) -> str:
    """Tra trạng thái đơn hàng theo mã đơn, ví dụ DH1024."""
    return {"DH1024": "đang giao, dự kiến 6/11"}.get(order_id, f"Không tìm thấy {order_id}")

class State(TypedDict):
    messages: Annotated[list, add_messages]          # reducer: nối thêm message mới

llm = ChatAnthropic(model="claude-haiku-5-5").bind_tools([get_order_status])
graph = StateGraph(State)
graph.add_node("agent", lambda s: {"messages": [llm.invoke(s["messages"])]})
graph.add_node("tools", ToolNode([get_order_status]))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)     # có gọi tool thì sang "tools", không thì kết thúc
graph.add_edge("tools", "agent")
app = graph.compile()
out = app.invoke({"messages": [("user", "Đơn DH1024 của tôi tới đâu rồi?")]}, {"recursion_limit": 10})

## Bài tập 5: Agent không dùng framework {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json, anthropic
client = anthropic.Anthropic()

def run_agent(question, tools, funcs, max_steps=8, model="claude-haiku-5-5"):
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        resp = client.messages.create(model=model, max_tokens=1000, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        calls = [b for b in resp.content if b.type == "tool_use"]
        if not calls:
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for c in calls:
            try:
                out = funcs[c.name](**c.input)
                results.append({"type": "tool_result", "tool_use_id": c.id, "content": json.dumps(out, ensure_ascii=False)})
            except Exception as e:
                results.append({"type": "tool_result", "tool_use_id": c.id, "content": f"Lỗi: {e}", "is_error": True})
            print(f"[bước {step}] {c.name}({c.input}) → {results[-1]['content'][:120]}")
        messages.append({"role": "user", "content": results})
    return "Dừng: vượt quá số bước cho phép."